## Step 4 -- Vicon ground truth for the confirmation sequences

Downloads the three subject archives (small; skipped if already on Drive), extracts `gt_skel_gbl_pos.txt` of the
nine sequences and saves `vicon/S{s}_{seq}.npz` with `pos` (T, 21, 3, inches) and `names` (21,), the format the
analysis scripts read. Also copies `calibration.cal`.


In [ ]:
import tarfile, io, numpy as np
VDIR = os.path.join(OUT_DIR, 'vicon'); os.makedirs(VDIR, exist_ok=True)
for u in [BASE + 'calibration.cal', BASE + 'calibrationreadme.txt']:
    dst = os.path.join(OUT_DIR, os.path.basename(u))
    if not os.path.exists(dst): fetch(u, dst)

def parse_pos(text):
    lines = [l for l in text.splitlines() if l.strip()]
    names = [n for n in lines[0].strip().split('\t') if n]
    rows = []
    for l in lines[1:]:
        f = [x for x in l.strip().split('\t') if x.strip()]
        rows.append([[float(v) for v in x.split()] for x in f])
    pos = np.asarray(rows, np.float32)
    assert pos.shape[1:] == (len(names), 3), (pos.shape, len(names))
    return pos, np.asarray(names)

for subj in sorted({s for s, _ in SUBSET}):
    arc = os.path.join(OUT_DIR, f'{subj.lower()}_vicon_pos_ori.tar.gz')
    if not os.path.exists(arc):
        fetch(BASE + f'vicon/{subj.lower()}_vicon_pos_ori.tar.gz', arc)
    with tarfile.open(arc) as tf:
        members = {m.name.lower(): m for m in tf.getmembers() if m.name.lower().endswith('gt_skel_gbl_pos.txt')}
        for s_, q_ in SUBSET:
            if s_ != subj: continue
            hit = [m for n, m in members.items() if f'/{q_}/' in '/' + n or f'{q_}/' in n]
            assert hit, f'no gt_skel_gbl_pos.txt for {s_} {q_}; members e.g. {list(members)[:5]}'
            pos, names = parse_pos(tf.extractfile(hit[0]).read().decode('utf-8', 'replace'))
            np.savez(os.path.join(VDIR, f'{s_}_{q_}.npz'), pos=pos, names=names)
            print(f'{s_}_{q_}: {pos.shape[0]} frames, {len(names)} joints')
print(sorted(os.listdir(VDIR)))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, glob, json, time, shutil, subprocess, getpass

SUBSET_NAME = 'confirm'   # S1-S3 x walking3 / acting2 / freestyle2 (held-out confirmation)
OUT_DIR = '/content/drive/MyDrive/MIST_TC_keypoints_confirm'   # private
LOCAL = '/content/tc'                                          # Colab local disk, wiped per file
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)

SUBSETS = {
    'confirm': [(s, q) for s in ['S1', 'S2', 'S3'] for q in ['walking3', 'acting2', 'freestyle2']],
}
# You type these yourself; they stay in this Colab runtime only.
TC_USER = input('TotalCapture username: ')
TC_PASS = getpass.getpass('TotalCapture password: ')
print(shutil.disk_usage('/content'))


## Step 1 -- probe the download links (send this output back)

Paste into `URLS` the download links you see on the TotalCapture site after logging in: one **video** link
(e.g. one subject/sequence), the **calibration** link, and one **Vicon ground-truth** link
(or the URL of the download page itself -- its links are listed).
The cell only sends HEAD requests and reads index pages -- nothing large is downloaded.

In [ ]:
import requests
URLS = [
    'https://cvssp.org/data/totalcapture/data/',   # HTTP Basic auth; lists every download link
]
for u in URLS:
    for auth in (None, (TC_USER, TC_PASS)):
        try:
            r = requests.head(u, auth=auth, allow_redirects=True, timeout=30)
            print(('auth ' if auth else 'anon ') + str(r.status_code), u)
            print('   ', {k: r.headers.get(k) for k in ('Content-Type', 'Content-Length', 'WWW-Authenticate', 'Content-Disposition')})
            if r.ok: break
        except Exception as e:
            print('ERR', u, e)
    if 'html' in (r.headers.get('Content-Type') or '') or u.endswith('/'):
        page = requests.get(u, auth=(TC_USER, TC_PASS), timeout=30).text
        links = re.findall(r'href=["\']([^"\']+)', page)
        print('    links:', len(links)); print('    ' + '\n    '.join(links[:80]))

## Step 2 -- 2D pose (MediaPipe Tasks, BlazePose full -> COCO-19, same mapping as the AP3D run)

In [ ]:
!pip -q install mediapipe==0.10.30
!wget -q -O /content/pose_landmarker_full.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task

In [ ]:
import numpy as np, cv2
import mediapipe as mp
from mediapipe.tasks import python as mpt
from mediapipe.tasks.python import vision
from concurrent.futures import ProcessPoolExecutor

MODEL = '/content/pose_landmarker_full.task'
_MID = {0: (11, 12), 2: (23, 24)}
_DIRECT = {1: 0, 3: 11, 4: 13, 5: 15, 6: 23, 7: 25, 8: 27, 9: 12, 10: 14, 11: 16,
           12: 24, 13: 26, 14: 28, 15: 2, 16: 7, 17: 5, 18: 8}
NAME = re.compile(r'(?i)(?<![a-z])s(?P<subj>\d).*?(?P<seq>rom\d|walking\d|freestyle\d|acting\d).*?cam[\W_]?(?P<cam>\d+)')

def to_coco19(lm):
    out = np.full((19, 2), np.nan)
    for j, s in _DIRECT.items(): out[j] = lm[s]
    for j, (a, b) in _MID.items(): out[j] = 0.5 * (lm[a] + lm[b])
    return out

def out_name(path):
    m = NAME.search(path.replace(os.sep, '/'))
    assert m, f'cannot parse subject/sequence/camera from {path}'
    return os.path.join(OUT_DIR, f"S{m['subj']}_{m['seq'].lower()}_cam{int(m['cam'])}.npz")

def run_video(path, min_vis=0.3):
    out = out_name(path)
    if os.path.exists(out):
        return out, 'skip'
    opts = vision.PoseLandmarkerOptions(
        base_options=mpt.BaseOptions(model_asset_path=MODEL),
        running_mode=vision.RunningMode.VIDEO, num_poses=1,
        min_pose_detection_confidence=0.5, min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5)
    cap = cv2.VideoCapture(path); W, H, fps = cap.get(3), cap.get(4), cap.get(5) or 60.0
    xy, vis, i = [], [], 0
    with vision.PoseLandmarker.create_from_options(opts) as det:
        while True:
            ok, bgr = cap.read()
            if not ok: break
            img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            res = det.detect_for_video(img, int(round(i * 1000.0 / fps)))
            lm = np.full((33, 2), np.nan); v = np.zeros(33)
            if res.pose_landmarks:
                for k, p in enumerate(res.pose_landmarks[0]):
                    v[k] = p.visibility
                    if p.visibility >= min_vis: lm[k] = (p.x * W, p.y * H)
            xy.append(to_coco19(lm)); vis.append(v); i += 1
    cap.release()
    tmp = out[:-4] + '.part.npz'   # write-then-rename: an interrupted run never leaves a truncated file
    np.savez(tmp, xy=np.asarray(xy, np.float32), vis=np.asarray(vis, np.float32), fps=fps, W=W, H=H,
             src=os.path.basename(path))
    os.replace(tmp, out)
    return out, f'{i} frames @ {fps:.2f} fps {int(W)}x{int(H)}'

## Step 3a -- discover the video files (Apache listing of `dataset/video/sN/`) and pick the subset

Prints every file with its size, then the selected `JOBS` and their total size. Send this output back
if the selection looks wrong (e.g. 0 jobs).

In [ ]:
import requests
from urllib.parse import urljoin
BASE = 'https://cvssp.org/data/totalcapture/data/dataset/'
AUTH = (TC_USER, TC_PASS)
SUBSET = SUBSETS[SUBSET_NAME]

def crawl(url, depth=0):
    r = requests.get(url, auth=AUTH, timeout=60); r.raise_for_status()
    out = []
    for h in re.findall(r'href="([^"?][^"]*)"', r.text):
        if h.startswith('/') or h.startswith('..'):
            continue                                    # parent directory link
        u = urljoin(url, h)
        out += crawl(u, depth + 1) if h.endswith('/') and depth < 4 else [u]
    return out

def size_gb(u):
    return int(requests.head(u, auth=AUTH, timeout=60).headers.get('Content-Length') or 0) / 1e9

files = crawl(BASE + 'video/')
print(len(files), 'files under video/')
for u in files[:40]:
    print(f'  {size_gb(u):7.2f} GB  {u[len(BASE):]}')
want = {(s.lower(), q) for s, q in SUBSET}
def key(u):
    p = u[len(BASE):].lower()
    s = re.search(r'(?<![a-z])s(\d)', p); q = re.search(r'(rom\d|walking\d|freestyle\d|acting\d)', p)
    return (f's{s.group(1)}', q.group(1)) if s and q else None
JOBS = sorted({u for u in files if key(u) in want and u.endswith(('.tar.gz', '.tgz', '.tar', '.zip'))})
total = sum(size_gb(u) for u in JOBS)
print(f'\n{len(JOBS)} jobs selected, {total:.1f} GB to stream')
for u in JOBS: print('  ', u[len(BASE):])

## Step 3b -- stream: download -> extract videos -> pose -> delete

Runs the whole subset in one go (one archive per sequence, 8 cameras each). Re-running skips videos
already on Drive, so a disconnected runtime can simply be restarted from Step 0.

In [ ]:
LIMIT = 10**6

def fetch(url, dst):
    subprocess.run(['wget', '-q', '-c', '--user', TC_USER, '--password', TC_PASS, '-O', dst, url], check=True)

def unpack(f, where):
    if f.endswith('.zip'): subprocess.run(['unzip', '-q', '-o', f, '-d', where], check=True)
    elif re.search(r'\.(tar(\.gz)?|tgz)$', f): subprocess.run(['tar', '-xf', f, '-C', where], check=True)
    else: return [f]
    os.remove(f)
    return glob.glob(where + '/**/*', recursive=True)

done = 0
for url in JOBS:
    s, q = key(url)          # folder name carries subject/sequence even if inner file names do not
    if all(os.path.exists(f'{OUT_DIR}/S{s[1:]}_{q}_cam{c}.npz') for c in range(1, 9)):
        print('skip (all 8 cams on Drive)', url); continue
    work = os.path.join(LOCAL, 'job', f'{s}_{q}'); shutil.rmtree(work, ignore_errors=True); os.makedirs(work)
    f = os.path.join(work, os.path.basename(url.split('?')[0]))
    t0 = time.time(); fetch(url, f); print(f'download {time.time()-t0:.0f}s  {os.path.getsize(f)/1e9:.2f} GB  {url}')
    vids = sorted(p for p in unpack(f, work) if p.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')))
    todo = [v for v in vids if not os.path.exists(out_name(v))][:max(0, LIMIT - done)]
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
        for out, msg in ex.map(run_video, todo):
            print(f'  {time.time()-t0:6.0f}s  {os.path.basename(out)}  {msg}')
    done += len(todo)
    shutil.rmtree(work, ignore_errors=True)
    if done >= LIMIT: break
print(len(glob.glob(OUT_DIR + '/*.npz')), 'keypoint files in', OUT_DIR)

## Step 4 -- small files: calibration + Vicon ground truth

Already saved by the test run (the Vicon archives contain every sequence of each subject); only needed for a fresh Drive.

In [ ]:
SMALL = [BASE + 'calibration.cal', BASE + 'calibrationreadme.txt'] +         [BASE + f'vicon/s{i}_vicon_pos_ori.tar.gz' for i in range(1, 6)]   # 3D joints, inches
for u in SMALL:
    print(f'{size_gb(u):.3f} GB  {u[len(BASE):]}')
    fetch(u, os.path.join(OUT_DIR, os.path.basename(u)))
print(os.listdir(OUT_DIR))

## Step 5 -- check: expect 8 cameras for every sequence of the subset

In [ ]:
import collections
have = collections.Counter(re.sub(r'_cam\d+\.npz$', '', os.path.basename(f))
                           for f in glob.glob(OUT_DIR + '/*.npz') if '.part' not in f)
for s_, q_ in SUBSET:
    v = os.path.exists(os.path.join(OUT_DIR, 'vicon', f'{s_}_{q_}.npz'))
    print(f'{s_}_{q_}: {have.get(f"{s_}_{q_}", 0)} / 8 cameras, vicon {"ok" if v else "MISSING"}')
print(sum(have.values()), '/', 8 * len(SUBSET), 'keypoint files in', OUT_DIR)
print('Then download the folder MIST_TC_keypoints_confirm (keypoints + vicon/) and place it as')
print('data/tc/MIST_TC_keypoints_confirm and data/tc/vicon (merge vicon/*.npz) in the repository.')
